# 12 · Test of H9: is the demonstrated approach to the bowl what the policy cannot copy?

**Evidence (notebook 11).** Our policies fail at the grasp: in failed episodes the gripper first closes a median 7-8 cm from the bowl's axis (the rim is at 5.6 cm; the teacher closes at 5.1 cm, the official policy at 4.6 cm), then closes again 3-7 times without lifting. The object-relative teacher reaches the rim along the last 15 cm of my hand's path, bent to end at the grasp point and skimming just above the rim; the moment it closes varies a lot from episode to episode (spread 24 steps). The official teleop policy comes down onto the rim and closes at a nearly fixed moment (spread 2.5 steps). In physics-only checks the same plans with a straight-down approach succeed 99% (vs 95%) and close 4.4 cm from the axis, with a spread of 1.2 steps.

**Experiment.** Regenerate on exactly the scenes and seeds of notebook 09 (5 consistent demos, 130 scenes each, cut to 450 successes), changing only the plan, and train / evaluate exactly as before (20k steps):
* `vertical`: object-relative (my grasp point, lift and carry shape, place point), but the gripper comes straight down from 8 cm above the grasp point instead of following my approach.
* `intent`: only my grasp side and place point; the whole motion is the scripted pick-and-place (the pipeline's ceiling: same generator, controller and dataset format, no human motion).
Baseline: notebook 09 `clean` (my approach) 67% / notebook 10 at 20k 69%.

**Predictions, fixed before running** (pooled over 2 evaluations, n = 100):
* H9 confirmed: `vertical` >= 83% (baseline + 15). Then the rest of my motion transfers and only the approach should be planned.
* H9 rejected: `vertical` <= 72%.
* `intent` tells where the remaining gap is: if `intent` is also <= ~75%, the limit is the scripted-teacher data itself, not the human motion; if `intent` >= 90% while `vertical` stays low, other parts of my motion (lift, carry, place) also hurt.

**Runtime:** T4, about 3 h per variant. Run once with `VARIANT = "vertical"` and once with `"intent"`, in two runtimes in parallel.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
VARIANT = "vertical"       # "vertical" or "intent": run this notebook twice, once with each
STRATEGY = {"vertical": "object_relative_vertical", "intent": "intent"}[VARIANT]
DEMOS = "f9115215f9_lefthand,c5f525e833,9a63cb1538_lefthand,92e0b86bc6_lefthand,65ee6ec786"   # grasp sides 106-116°
PER_DEMO, N_EPISODES, SEED0 = 130, 450, 2_000_000     # the same seeds as notebook 09: same scenes, same perturbations
HUB_REPO, SEED, STEPS, BATCH = "phone2panda-data", 0, 20000, 32
NAME = f"ours_consistent_{VARIANT}"
RUN = f"act_{NAME}_s{SEED}"
GEN = f"{BASE}/gen_{VARIANT}"
WORKERS = os.cpu_count()
from p2p.sim.env import render_speed
r = render_speed(); print(r)
assert r["gpu_rendering"], "rendering is on the CPU: Runtime -> Change runtime type -> T4 GPU, then restart"

## Generate (resumable: rerun the cell after a disconnect and it continues)

In [ ]:
!python -m p2p.data.generate data/processed {GEN} --strategy {STRATEGY} --demos {DEMOS} --per-demo {PER_DEMO} --seed0 {SEED0} --workers {WORKERS}

In [ ]:
import pandas as pd
log = pd.read_json(f"{GEN}/generation_log.jsonl", lines=True).drop_duplicates("seed", keep="last")
print(VARIANT, "yield", round(log.success.mean(), 3), "successes", int(log.success.sum()), "of", len(log))
display(log.groupby("demo").success.agg(["mean", "sum"]))
assert log.success.sum() >= N_EPISODES, "fewer successes than N_EPISODES: lower N_EPISODES in BOTH variants"

## Dataset: 450 successes spread evenly over the demos, as in notebook 09

In [ ]:
!python -m p2p.data.lerobot_io ours {GEN} {BASE}/lerobot/{NAME} --balanced {N_EPISODES}
!python tools/demo_consistency.py {BASE}/lerobot/{NAME}
from p2p.hub import push
push(f"{BASE}/lerobot/{NAME}", HUB_REPO, path_in_repo=f"lerobot/{NAME}")

## Train and evaluate (same settings as every other run; evaluation now also records behaviour)

In [ ]:
!python -m p2p.policy.act train {BASE}/lerobot/{NAME} {BASE}/runs/{RUN} --steps {STEPS} --batch-size {BATCH} --seed {SEED}

In [ ]:
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 6 --out {BASE}/runs/{RUN}/eval
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --out {BASE}/runs/{RUN}/eval_rerun
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --temporal-ensemble 0.01 --out {BASE}/runs/{RUN}/eval_te
import glob
evals = " ".join(sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")))
!python tools/policy_failures.py {evals}
for e in sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")):
    print(e.split("/")[-2], json.dumps(json.load(open(e))["behaviour"]))

In [ ]:
push(f"{BASE}/runs/{RUN}", HUB_REPO, path_in_repo=f"runs/{RUN}")
!cp {GEN}/generation_log.jsonl {BASE}/runs/{RUN}/generation_log.jsonl
!cd {BASE}/runs && zip -qr {BASE}/{RUN}.zip {RUN} -x "*/final/*" "*/step_*"
try:
    from google.colab import files; files.download(f"{BASE}/{RUN}.zip")
except Exception:
    print("download", f"{BASE}/{RUN}.zip", "from the Output panel")